# 11.9 縮圖與裁切會丟掉什麼？

# 第 11 章：對齊、問答與語言能力保留

前置：第10章介面與第7章SFT。能力實驗需先有「文字屬性→正確答案」基模，以及可分類的視覺encoder；用scripts/prepare_data.py和pretrain_encoders.py準備。下面隨機模型只驗證資料／梯度，所有品質留給讀者訓練後量測。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：圖片切小方塊：先檢查能不能拼回來**

16×16 RGB，patch邊長4，得到16個patch。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/patchify.svg")))

**先想一想：**物件在最右邊，中心裁切還能回答它的顏色嗎？

縮圖可能讓小物件消失；中心裁切可能丟邊緣物件。前處理不是中性的I/O，會改變任務是否有足夠資訊。

**把直覺寫成數學：**可見資訊先檢查再訓練；不存在的像素無法由更好的projector還原。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.multimodal import scene

image = scene("red", "square", offset=7)
crop = image[:, 4:12, 4:12]
print("原圖紅色像素", int((image[0] > 0).sum()), "裁切後", int((crop[0] > 0).sum()))

**如何讀結果：**這是生成規則下的圖像資訊檢查；另外保存前處理設置與原圖以便定位錯誤。

**只改一件事：**只移動物件offset，不改模型。
